# Phase IV — Ensemble Methods
## Bank Marketing Dataset — ML Lab Project 2026

**Authors:** Arman Bazarchi, Ines Maria Madeira Prates

---

**Prerequisites:** Run `preprocessing.ipynb` first to generate the processed data in `data/`.

This notebook trains tree-based ensembles — **Random Forest** (bagging) and two **boosting** models
(AdaBoost, Gradient Boosting) — and compares them on three axes: **performance**, **training time**,
and **feature importance**. Unlike the kernel SVMs of Phase III, tree ensembles scale comfortably,
so here we train on the **full 33k** training set (no subsample); trees are also scale-invariant, so
the standardized features do no harm.

### Outline
1. Load Processed Data & Prepare Features
2. Baseline & Imbalance Handling
3. Train the Ensembles (Random Forest, AdaBoost, Gradient Boosting)
4. Comparison — Performance & Training Time
5. Feature Importance (impurity & permutation)
6. Best Ensemble — Detailed Evaluation
7. Summary & Discussion

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import time

from sklearn.model_selection import train_test_split
from sklearn.ensemble import (RandomForestClassifier, AdaBoostClassifier,
                              GradientBoostingClassifier)
from sklearn.dummy import DummyClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             average_precision_score, confusion_matrix,
                             roc_curve, precision_recall_curve, classification_report)

warnings.filterwarnings('ignore')
np.random.seed(42)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

---
## 1. Load Processed Data & Prepare Features

Same feature pipeline and split as Phases II-III so results are directly comparable: drop the leaky
`duration`, one-hot encode the categoricals, same 80/20 stratified split. Trees are scale-invariant
(they split on thresholds), so the standardized numerics are harmless; we keep them for a clean
apples-to-apples comparison with the earlier phases.

In [2]:
X_scaled_df = pd.read_csv('data/X_scaled.csv')       # 15 standardized numeric features
cat_features = pd.read_csv('data/cat_features.csv')   # job, marital, month, day_of_week
y = pd.read_csv('data/y.csv')['y']

# Drop leaky 'duration'; one-hot the categoricals (drop_first avoids the dummy trap)
X_num = X_scaled_df.drop(columns=['duration'])
X_cat = pd.get_dummies(cat_features, drop_first=True).astype(int)
X = pd.concat([X_num.reset_index(drop=True), X_cat.reset_index(drop=True)], axis=1)
feature_names = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Full design matrix: {X.shape}")
print(f"Train: {X_train.shape[0]} samples ({y_train.mean()*100:.2f}% positive)")
print(f"Test:  {X_test.shape[0]} samples ({y_test.mean()*100:.2f}% positive)")
print("Trees scale fine -> we train on the FULL training set (no subsample needed, unlike Phase III).")

Full design matrix: (41188, 41)
Train: 32950 samples (11.27% positive)
Test:  8238 samples (11.26% positive)
Trees scale fine -> we train on the FULL training set (no subsample needed, unlike Phase III).


---
## 2. Baseline & Imbalance Handling

We keep the same stratified dummy baseline, and address the 11% imbalance inside each model:
Random Forest supports `class_weight='balanced'`; AdaBoost / Gradient Boosting do not, so we pass
**balanced `sample_weight`** to their `fit` (the equivalent re-weighting).

In [3]:
dummy = DummyClassifier(strategy='stratified', random_state=42).fit(X_train, y_train)
dummy_pred = dummy.predict(X_test)
dummy_scores = dummy.predict_proba(X_test)[:, 1]
print("Baseline (stratified dummy):")
print(f"  Accuracy: {accuracy_score(y_test, dummy_pred):.4f}   F1: {f1_score(y_test, dummy_pred):.4f}")
print(f"  ROC-AUC:  {roc_auc_score(y_test, dummy_scores):.4f}   AUPR: {average_precision_score(y_test, dummy_scores):.4f}")

# Balanced sample weights for the boosters (which lack a class_weight parameter)
sw_train = compute_sample_weight('balanced', y_train)
print(f"\nBalanced sample weights: minority rows up-weighted to {sw_train.max():.2f}, majority to {sw_train.min():.2f}")

Baseline (stratified dummy):
  Accuracy: 0.8037   F1: 0.1207
  ROC-AUC:  0.5051   AUPR: 0.1137

Balanced sample weights: minority rows up-weighted to 4.44, majority to 0.56


---
## 3. Train the Ensembles

- **Random Forest** (bagging): many deep trees on bootstrap samples, votes averaged → low variance.
- **AdaBoost** (boosting): sequential shallow stumps, each focusing on the previous errors.
- **Gradient Boosting** (boosting): sequential shallow trees fit to the gradient of the loss;
  `subsample=0.8` makes it stochastic (a little regularization).

We time each fit on the full 33k and evaluate on the test set with `predict_proba` for ROC-AUC/AUPR.

In [ ]:
def eval_model(name, model, Xte, yte, fit_time=None):
    scores = model.predict_proba(Xte)[:, 1]
    pred = model.predict(Xte)
    return {'model': name,
            'accuracy': accuracy_score(yte, pred),
            'f1': f1_score(yte, pred),
            'roc_auc': roc_auc_score(yte, scores),
            'aupr': average_precision_score(yte, scores),
            'fit_time_s': fit_time}

models, rows = {}, []

# Random Forest (bagging) -- class_weight handles imbalance; OOB gives a free validation estimate
rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight='balanced',
                            oob_score=True, n_jobs=-1, random_state=42)
t0 = time.perf_counter(); rf.fit(X_train, y_train); rf_t = time.perf_counter() - t0
models['Random Forest'] = rf

# AdaBoost (boosting) -- balanced sample weights
ada = AdaBoostClassifier(n_estimators=300, learning_rate=0.5, random_state=42)
t0 = time.perf_counter(); ada.fit(X_train, y_train, sample_weight=sw_train); ada_t = time.perf_counter() - t0
models['AdaBoost'] = ada

# Gradient Boosting (boosting) -- balanced sample weights, stochastic (subsample<1)
gb = GradientBoostingClassifier(n_estimators=300, learning_rate=0.1, max_depth=3,
                                subsample=0.8, random_state=42)
t0 = time.perf_counter(); gb.fit(X_train, y_train, sample_weight=sw_train); gb_t = time.perf_counter() - t0
models['Gradient Boosting'] = gb

for (name, m), ft in zip(models.items(), [rf_t, ada_t, gb_t]):
    rows.append(eval_model(name, m, X_test, y_test, ft))
ens_df = pd.DataFrame(rows).set_index('model')

print("Ensembles trained on full 33k, evaluated on full test set:")
print(ens_df.round(4).to_string())
print(f"\nRandom Forest OOB accuracy (free validation): {rf.oob_score_:.4f}")

# All three ensembles CLEAR the ~0.79 ROC-AUC ceiling that every linear/SVM model hit:
# Gradient Boosting 0.811 and Random Forest 0.807, with AUPR ~0.48-0.49 (up from ~0.43 before).
# This is the first REAL jump in the project -- trees capture the feature interactions and
# threshold effects that linear/kernel models could not express.
#
# Random Forest is the all-rounder: best F1 (0.52), best AUPR (0.486), best accuracy (0.872), and
# the FASTEST fit (~1.2s). Its OOB accuracy (0.864) closely tracks the held-out accuracy (0.872),
# a good sign the model is well-calibrated and not overfit. Gradient Boosting edges ROC-AUC (0.811
# vs 0.807); AdaBoost trails both (0.797 / 0.461) -- its shallow stumps are the weakest learner here.

---
## 4. Comparison — Performance & Training Time

In [ ]:
compare = ens_df.copy()
compare.loc['Baseline (dummy)'] = [accuracy_score(y_test, dummy_pred), f1_score(y_test, dummy_pred),
                                   roc_auc_score(y_test, dummy_scores),
                                   average_precision_score(y_test, dummy_scores), np.nan]
compare = compare.sort_values('roc_auc', ascending=False)
print("Performance & training time (full test set):")
print(compare.round(4).to_string())

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
metric_df = compare[['accuracy', 'f1', 'roc_auc', 'aupr']]
metric_df.plot(kind='bar', ax=axes[0], edgecolor='black')
axes[0].set_title('Ensembles vs Baseline -- Test Metrics', fontweight='bold')
axes[0].set_ylabel('Score'); axes[0].set_xlabel(''); axes[0].tick_params(axis='x', rotation=20)
axes[0].legend(loc='upper right', ncol=2)

ens_df['fit_time_s'].plot(kind='bar', ax=axes[1], color='slateblue', edgecolor='black')
axes[1].set_title('Training Time (full 33k)', fontweight='bold')
axes[1].set_ylabel('seconds'); axes[1].set_xlabel(''); axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

# PERFORMANCE: Gradient Boosting and Random Forest lead (ROC-AUC ~0.81, AUPR ~0.48), clearly above
# the Phase II logistic (0.797 / 0.428) and Phase III SVMs (~0.793 / ~0.43). The absolute gain is
# modest (~+0.015 ROC-AUC, ~+0.05 AUPR) but it is the FIRST model family to break the overlap-driven
# plateau -- and the AUPR lift (which matters most under imbalance) is the largest of the project.
#
# TRAINING TIME: the striking contrast with Phase III -- all three ensembles fit the FULL 33k in
# 1-5s (RF fastest ~1.2s thanks to parallel trees; GB slowest ~5s because boosting is sequential),
# whereas the kernel SVMs needed a 12k subsample and ~55s just for the linear grid. For this dataset
# the ensembles dominate on BOTH accuracy-type metrics AND speed -- the practical winners.

---
## 5. Feature Importance

We look at importance two ways:
- **Impurity-based** (`feature_importances_`): fast, but biased toward high-cardinality / continuous
  features.
- **Permutation importance** (on the test set): shuffles each feature and measures the ROC-AUC drop
  — slower but unbiased and model-agnostic. This is the more trustworthy view.

In [ ]:
# Impurity-based importance: Random Forest vs Gradient Boosting
imp = pd.DataFrame({'RandomForest': rf.feature_importances_,
                    'GradientBoosting': gb.feature_importances_}, index=feature_names)
imp = imp.reindex(imp.mean(axis=1).sort_values(ascending=False).index)
print("Impurity-based feature importance (top 15):")
print(imp.head(15).round(4).to_string())

imp.head(12).iloc[::-1].plot(kind='barh', figsize=(12, 8), edgecolor='black')
plt.title('Impurity-based Feature Importance (top 12)', fontweight='bold')
plt.xlabel('Importance'); plt.tight_layout()
plt.show()

# Impurity importance is dominated by the macro block (nr.employed, euribor3m) for both models --
# Gradient Boosting concentrates almost half its weight on nr.employed alone (0.45), while Random
# Forest spreads importance more evenly. The same economy + campaign signal (emp.var.rate, campaign,
# poutcome, cons.conf.idx) follows, consistent with Phase I-III.
#
# Watch out for 'age': RF ranks it 3rd (0.11) -- but see the permutation importance next. age is a
# continuous feature with many possible split points, so trees "use" it often; impurity importance
# is biased toward exactly such high-cardinality features, which can overstate their real value.

In [ ]:
# Permutation importance on the test set for the best ensemble (unbiased, scoring = ROC-AUC)
best_name = ens_df['roc_auc'].idxmax()
best_model = models[best_name]
perm = permutation_importance(best_model, X_test, y_test, scoring='roc_auc',
                              n_repeats=10, n_jobs=-1, random_state=42)
perm_imp = pd.Series(perm.importances_mean, index=feature_names).sort_values(ascending=False)

print(f"Permutation importance ({best_name}, ROC-AUC drop when shuffled, top 15):")
print(perm_imp.head(15).round(4).to_string())

perm_imp.head(12).iloc[::-1].plot(kind='barh', figsize=(12, 8), color='teal', edgecolor='black')
plt.title(f'Permutation Importance -- {best_name} (top 12)', fontweight='bold')
plt.xlabel('Mean ROC-AUC drop when feature shuffled'); plt.tight_layout()
plt.show()

# Permutation importance (the unbiased view) confirms nr.employed and euribor3m as the top drivers
# (each ~0.034 ROC-AUC drop when shuffled), then contact, campaign and poutcome -- the economy +
# campaign signal we have tracked since Phase I.
#
# The key contrast: 'age' -- ranked 3rd by RF impurity (0.11) -- does NOT appear in the permutation
# top 15; shuffling it barely changes ROC-AUC. This is exactly the impurity-importance bias: trees
# split on age frequently (it is continuous) but it carries little real predictive signal. The two
# methods agreeing on nr.employed/euribor3m but disagreeing on age is a textbook reminder to trust
# permutation importance for conclusions -- and it lines up with the weak age-vs-target correlation
# (r=0.03) we found in Phase I's EDA.

---
## 6. Best Ensemble — Detailed Evaluation

In [ ]:
best_scores = best_model.predict_proba(X_test)[:, 1]
best_pred = best_model.predict(X_test)
print(f"Best ensemble by test ROC-AUC: {best_name}")
print("=" * 60)
print(classification_report(y_test, best_pred, target_names=['No', 'Yes']))

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
cm = confusion_matrix(y_test, best_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No', 'Yes'], yticklabels=['No', 'Yes'])
axes[0].set_title(f'{best_name} -- Confusion Matrix', fontweight='bold')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')

fpr, tpr, _ = roc_curve(y_test, best_scores)
axes[1].plot(fpr, tpr, color='steelblue', label=f'AUC = {roc_auc_score(y_test, best_scores):.3f}')
axes[1].plot([0, 1], [0, 1], '--', color='gray')
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve', fontweight='bold'); axes[1].legend()

prec, rec, _ = precision_recall_curve(y_test, best_scores)
axes[2].plot(rec, prec, color='darkorange', label=f'AUPR = {average_precision_score(y_test, best_scores):.3f}')
axes[2].axhline(y=y_test.mean(), linestyle='--', color='gray', label=f'baseline = {y_test.mean():.3f}')
axes[2].set_xlabel('Recall'); axes[2].set_ylabel('Precision')
axes[2].set_title('Precision-Recall Curve', fontweight='bold'); axes[2].legend()

plt.suptitle(f'{best_name} -- Performance on Full Test Set', fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

# Gradient Boosting is the best ensemble (ROC-AUC 0.811). It recovers 66% of subscribers (recall)
# at 38% precision -- a better recall/precision balance than any linear or kernel model (F1 0.48 vs
# the ~0.46 they plateaued at). The confusion matrix and curves show the familiar wide-net profile,
# but shifted modestly outward: the ensemble genuinely separates the classes a little better,
# pushing past the ~0.79 ROC-AUC / ~0.43 AUPR ceiling for the first time in the project.

---
## 7. Summary & Discussion

### Key Findings — Phase IV

**Setup:**
- Same features/split as Phases II–III. Tree ensembles scale, so we trained on the **full 33k** (no
  subsample); trees are scale-invariant. Imbalance handled via `class_weight='balanced'` (RF) and
  balanced `sample_weight` (the boosters).

**Performance — the first real jump:**
- **Gradient Boosting** (ROC-AUC 0.811, AUPR 0.484) and **Random Forest** (0.807, 0.486) clear the
  **~0.79 ROC-AUC / ~0.43 AUPR ceiling** that every linear and kernel model hit. RF also wins F1
  (0.52) and accuracy (0.872); **AdaBoost** trails (0.797 / 0.461).
- The gain is modest in absolute terms (~+0.015 ROC-AUC, ~+0.05 AUPR over the best Phase II/III
  model) but it is the **first family to break the plateau** — trees capture the feature
  interactions and threshold effects that linear/kernel models could not.

**Training time:**
- All three fit the full 33k in **1–5s** (RF ~1.2s, parallel; GB ~5s, sequential) — versus the
  kernel SVMs that needed a 12k subsample and ~55s for the linear grid alone. Ensembles are by far
  the most scalable family here, and RF wins on speed *and* most metrics.

**Feature importance:**
- Both views agree the **macro block (nr.employed, euribor3m)** plus the **campaign signal
  (contact, campaign, poutcome)** drive predictions — consistent with Phases I–III.
- Impurity vs permutation **disagree on `age`**: RF impurity ranks it 3rd, but permutation shows
  shuffling it barely moves ROC-AUC — the classic impurity bias toward continuous features.
  Permutation importance is the trustworthy one (and matches age's r=0.03 from EDA).

**Dataset challenges:**
- Even the best ensemble plateaus around ROC-AUC 0.81 / AUPR 0.48: the residual class overlap
  (rooted in the excluded leaky `duration`) still caps performance — ensembles **raise** the
  ceiling a little but do not eliminate it.

**Where we stand across all phases:**
- Best model so far: **Gradient Boosting / Random Forest (~0.81 ROC-AUC)**, a clear step above
  logistic (0.797) and SVM (~0.793). Trees win on accuracy, F1, AUPR *and* training time — the
  practical choice for this dataset.

---

*Phase V will perform rigorous model selection and validation (nested CV) across all phases, with statistical tests, to choose and confirm the final model.*